# ASC27 world model reproduction

复现入口，未在本次打包时运行。正式测量来自AutoDL RTX 3090。使用GPU运行前先按README准备依赖和权重；Colab免费T4的显存和内存不保证能运行该模型。

## 1 挂载提交目录

在AutoDL/Jupyter上直接使用默认目录。在Colab中先将提交目录放入Drive，然后运行挂载并调整PACKAGE。

In [ ]:
from pathlib import Path
import os, subprocess
PACKAGE = Path("/root/ASC27_worldmodel")
# Colab example:
# from google.colab import drive
# drive.mount("/content/drive")
# PACKAGE = Path("/content/drive/MyDrive/ASC27_worldmodel")
assert (PACKAGE / "code/run_case.sh").is_file(), PACKAGE


## 2 环境与路径

首次迁移需安装GPU版本PyTorch及README所列依赖，并安装项目。权重路径必须指向实际下载的文件。

In [ ]:
import sys
PYTHON = "/root/autodl-tmp/miniconda3/envs/asc_wma/bin/python"
if not Path(PYTHON).exists():
    PYTHON = sys.executable
os.environ["PYTHON"] = PYTHON
os.environ["CKPT_PATH"] = "/root/autodl-tmp/checkpoints/UnifoLM-WMA-0-Dual/unifolm_wma_dual.ckpt"
os.environ["OPENCLIP_PRETRAINED_PATH"] = "/root/autodl-tmp/openclip_cache/open_clip_pytorch_model.bin"
os.environ["RUN_ROOT"] = str(PACKAGE.parent / "asc27_notebook_reproduction_runs")
subprocess.run([PYTHON, "-m", "pip", "install", "--no-deps", "-e", str(PACKAGE / "code/unifolm-world-model-action")], check=True)


## 3 先查看命令

本单元不运行GPU推理。完整组stackbox使用7轮，cleanup使用8轮；短测改为short_2s。

In [ ]:
MODE = "optimized"
SCENARIO = "unitree_z1_dual_arm_stackbox"
CASE_ID = "1"
GROUP = "full"
command = ["bash", str(PACKAGE / "code/run_case.sh"), MODE, SCENARIO, CASE_ID, GROUP]
subprocess.run(command + ["--dry-run"], check=True)


## 4 运行单case

需要GPU。RUN_INFERENCE默认关闭，以免无卡模式误启动；确认资源后设为True。

In [ ]:
RUN_INFERENCE = False
if RUN_INFERENCE:
    subprocess.run(command, check=True)
else:
    print("Inference disabled; set RUN_INFERENCE=True when the GPU environment is ready.")


## 5 查看已有提交汇总

In [ ]:
import json
summary = json.loads((PACKAGE / "results/summary.json").read_text())
for row in summary:
    print(row["scenario"], row["case_id"], "speedup", round(row["time_before_optim"] / row["time_after_optim"], 3), "PSNR", row["psnr"])
